# SilverWork_incremental_industrial_v2
Incremental Silver processing using only new Bronze rows.


## Step 1 — Imports and setup
This cell imports Spark, Window, and Delta helpers, switches to the right catalog, makes sure the **Silver schema** exists, and creates a `silver_run_id` for the current run.


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid
from pyspark.sql import Window

In [0]:
spark.sql("use catalog adb_datatocrunchproject")
spark.sql("create schema if not exists silver_schema")
silver_run_id = str(uuid.uuid4())
print("Current Silver Run ID: ", silver_run_id)

## Step 2 — Silver control table
This table stores the Silver processing history: one row is appended per entity per run. The latest SUCCESS row (by `updated_at`) is the current watermark.

It helps us track:
* the latest Bronze run already processed by Silver
* the latest Bronze ingestion timestamp already processed
* how many rows were merged in each Silver run


In [0]:
spark.sql("""
    create table if not exists adb_datatocrunchproject.silver_schema.processing_control(
        layer string,
        entity_name string,
        last_processed_bronze_run_id string,
        last_processed_bronze_ingested_at timestamp,
        rows_merged bigint,
        run_status string,
        silver_run_id string,
        updated_at timestamp
    )
    using delta
""")


## Step 3 — Helper functions
This cell contains reusable logic for Silver:

* `upsert_to_silver()` merges cleaned / transformed rows into the Silver target table
* `get_last_processed_bronze_ingested_at()` reads the Silver watermark
* `upsert_silver_control()` updates the Silver control table
* `get_incremental_bronze()` reads only new Bronze rows that Silver has not processed yet


In [0]:
def upsert_to_silver(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (dt.alias("target")
            .merge(df_source.alias("source"), f"target.{join_key} = source.{join_key}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
    else:
        df_source.write.saveAsTable(target_table)

In [0]:
def get_last_processed_bronze_ingested_at(entity_name:str):
    ctrl = (
        spark.table("adb_datatocrunchproject.silver_schema.processing_control")
        .filter(
            (F.col("layer") == "silver") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "SUCCESS")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )
    rows = ctrl.collect()
    if not rows:
        return None, None
    return (
        rows[0]["last_processed_bronze_ingested_at"],
        rows[0]["last_processed_bronze_run_id"]
    )

In [0]:
def upsert_silver_control(entity_name, last_processed_bronze_run_id, last_processed_bronze_ingested_at, rows_merged):
    ctrl_df = spark.createDataFrame(
        [
            (
                "silver",
                entity_name,
                last_processed_bronze_run_id,
                last_processed_bronze_ingested_at,
                int(rows_merged),
                "SUCCESS",
                silver_run_id,
                datetime.utcnow(),
            )
        ],
        schema="""
            layer string,
            entity_name string,
            last_processed_bronze_run_id string,
            last_processed_bronze_ingested_at timestamp,
            rows_merged bigint,
            run_status string,
            silver_run_id string,
            updated_at timestamp
        """
    )

    # Append one row per run so the control table keeps a full run history.
    # get_last_processed_bronze_ingested_at() reads the latest SUCCESS row by updated_at.
    ctrl_df.write.format("delta").mode("append").saveAsTable(
        "adb_datatocrunchproject.silver_schema.processing_control"
    )


In [0]:
def get_incremental_bronze(bronze_table, entity_name):
    last_ingested_at, last_run_id = get_last_processed_bronze_ingested_at(entity_name)
    bronze_df = spark.read.table(bronze_table)

    if last_ingested_at is None:
        return bronze_df, last_ingested_at, last_run_id

    return (
        bronze_df.filter(F.col("bronze_ingested_at") > F.lit(last_ingested_at)),
        last_ingested_at,
        last_run_id
    )


**Step 4 — Orders incremental processing**

This cell processes **orders** from Bronze to Silver.

It does the following:
* reads only new Bronze order rows
* cleans values like `order_status` and `order_amount`
* keeps only the latest version per `order_id`
* validates business rules

** • sends bad rows to quarantine
• merges good rows into orders_transformed


In [0]:
df_raw = spark.sql("select * from adb_datatocrunchproject.bronze_schema.orders_raw")
display(df_raw)


In [0]:
# Step 4 - Orders incremental processing
# Read only the Bronze order rows that Silver has not processed yet.
orders_inc, last_orders_ingested_at, last_order_run_id = get_incremental_bronze(
    "adb_datatocrunchproject.bronze_schema.orders_raw",
    "orders"
)

# Count the incremental order rows entering Silver in this run.
orders_inc_count = orders_inc.count()
print(f"orders rows_to_process_in_silver = {orders_inc_count}")

# Only run Silver order cleaning and validation when there are new Bronze order rows.
if orders_inc_count > 0:
    # Create a window that keeps the latest order record for each order_id.
    order_window = Window.partitionBy("order_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    # Start the Silver order-cleaning pipeline. This block standardizes and deduplicates raw order records.
    orders_cleaned = (
        orders_inc
        # Standardize order_status to uppercase so values such as shipped and SHIPPED become consistent.
        .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
        .withColumn("order_status", F.when(F.col("order_status") == "", F.lit(None)).otherwise(F.col("order_status")))
        # Remove formatting characters from order_amount so it can be cast to a numeric type.
        .withColumn("order_amount", F.regexp_replace(F.col("order_amount"), r"[$, ]", ""))
        .withColumn("order_amount", F.when(F.trim(F.col("order_amount")).isin("N/A", "NULL", "??", ""), None).otherwise(F.col("order_amount")))
        .withColumn("order_amount", F.col("order_amount").cast("double"))
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("updated_at", F.to_timestamp("updated_at"))
        # Assign a row number inside each business key so we can keep only the latest version of that record.
        .withColumn("row_rank", F.row_number().over(order_window))
        # Keep only the latest record for each business key.
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    # Apply Silver data-quality rules to the cleaned order records.
    orders_validated = (
        orders_cleaned
        .withColumn(
            "to_be_verified_by_orders_team",
            F.when(F.col("customer_id").isNull(), "verify_customer_id")
            .when(F.col("product_id").isNull(), "verify_product_id")
            .when(F.col("order_status").isNull() | (F.trim(F.col("order_status")) == ""), "verify_order_status")
            .when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0), "verify_order_amount")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_order_amount",
            F.when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0), F.lit(True))
            .otherwise(F.lit(False))
        )
        .withColumn("order_date", F.to_date("created_at"))
        .withColumn("order_year", F.year("created_at"))
        .withColumn("order_month", F.month("created_at"))
        .withColumn("order_day", F.dayofmonth("created_at"))
        .withColumn("order_dow", F.date_format("created_at", "E"))
    )

    # Keep only valid order rows for the transformed Silver table.
    orders_good = orders_validated.filter(F.col("to_be_verified_by_orders_team") == "No Issues")
    
    # Send invalid order rows to the quarantine dataset for manual review.
    orders_bad = (
        orders_validated
        .filter(F.col("to_be_verified_by_orders_team") != "No Issues")
        .withColumn("quarantine_ts", F.current_timestamp())
    )

    # Persist the cleaned and deduplicated orders (before validation) as their own Silver table.
    upsert_to_silver(
        orders_cleaned,
        "adb_datatocrunchproject.silver_schema.orders_cleaned",
        "order_id"
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(
        orders_good,
        "adb_datatocrunchproject.silver_schema.orders_transformed",
        "order_id"
    )

    # Append bad order rows to the quarantine table instead of losing them.
    orders_bad.write.format("delta").mode("append").saveAsTable(
        "adb_datatocrunchproject.silver_schema.orders_quarantine"
    )

    mx_ingested = orders_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = (
        orders_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )

    upsert_silver_control("orders", mx_run, mx_ingested, orders_good.count())

else:
    print("No new orders Bronze rows for Silver.")
    
    upsert_silver_control(
        "orders",
        last_order_run_id,
        last_orders_ingested_at,
        orders_inc_count
    )


In [0]:
# One-time backfill: the Bronze orders were already processed before orders_cleaned was persisted.
# orders_cleaned = good rows + quarantined rows, using only the cleaned (pre-validation) columns.
cleaned_cols = [
    "order_id", "customer_id", "product_id", "order_status", "order_amount",
    "created_at", "updated_at", "bronze_ingested_at", "bronze_run_id",
    "bronze_source_table", "silver_run_id"
]

orders_cleaned_backfill = (
    spark.table("adb_datatocrunchproject.silver_schema.orders_transformed").select(*cleaned_cols)
    .unionByName(
        spark.table("adb_datatocrunchproject.silver_schema.orders_quarantine").select(*cleaned_cols)
    )
    .dropDuplicates(["order_id"])
)

upsert_to_silver(
    orders_cleaned_backfill,
    "adb_datatocrunchproject.silver_schema.orders_cleaned",
    "order_id"
)

display(spark.table("adb_datatocrunchproject.silver_schema.orders_cleaned"))

In [0]:
%sql
select * from adb_datatocrunchproject.silver_schema.orders_quarantine

In [0]:
%sql
select * from adb_datatocrunchproject.silver_schema.orders_transformed

**
Step 5 — Products incremental processing
This cell processes products from Bronze to Silver.
It handles:
• product name cleanup
• category standardization
• price cleanup and numeric conversion
• latest-record selection per product_id
• data quality validation
• quarantine for bad rows
• merge into Silver current-state tables

In [0]:
%sql
select * from adb_datatocrunchproject.bronze_schema.products_raw;


In [0]:
# Step 5 - Products incremental processing
# Read only the Bronze product rows that Silver has not processed yet.
products_inc, last_products_ingested_at, last_products_run_id = get_incremental_bronze("adb_datatocrunchproject.bronze_schema.products_raw", "products")

# Count the incremental product rows entering Silver in this run.
products_inc_count = products_inc.count()
print(f"products rows_to_process_in_silver = {products_inc_count}")

if products_inc_count > 0:
    # Create a window that keeps the latest product record for each product_id.
    product_window = Window.partitionBy("product_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    # Start the Silver product-cleaning pipeline. This block standardizes and deduplicates raw product records.
    products_cleaned = (
        products_inc
        # Standardize product_name by trimming spaces and converting text to uppercase.
        .withColumn("product_name", F.upper(F.trim(F.col("product_name"))))
        .withColumn("product_name", F.regexp_replace(F.col("product_name"), r"[-_]", " "))  # New Logic
        .withColumn("product_name", F.when(F.col("product_name") == "", F.lit(None)).otherwise(F.col("product_name")))
        .withColumn(
            "category",
            F.when(F.upper(F.trim(F.col("category"))).contains("ELECTRNICS"), "ELECTRONICS")
            .when(F.upper(F.trim(F.col("category"))) == "FITNESS", "FITNESS")
            .when(F.upper(F.trim(F.col("category"))) == "LIFESTYLE", "LIFESTYLE")
            .otherwise(F.upper(F.trim(F.col("category"))))
        )
        # Start cleaning the product price field before converting it to numeric.
        .withColumn("price", F.trim(F.col("price")))
        .withColumn("price", F.regexp_replace(F.col("price"), r"\$", ""))
        .withColumn("price", F.regexp_replace(F.col("price"), r",", ""))
        .withColumn("price", F.regexp_replace(F.col("price"), r" ", ""))
        .withColumn("price", F.expr("try_cast(price as double)"))
        .withColumn("updated_at", F.to_timestamp("updated_at"))
        # Assign a row number inside each business key so we can keep only the latest version of that record.
        .withColumn("row_rank", F.row_number().over(product_window))
        # Keep only the latest record for each business key.
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(products_cleaned, "adb_datatocrunchproject.silver_schema.products_cleaned", "product_id")

    # Apply Silver data-quality rules to the cleaned product records.
    products_validated = (
        products_cleaned
        .withColumn(
            "to_be_verified_by_products_team",
            F.when(F.col("product_name").isNull(), "verify_product_name")
            .when(F.col("category").isNull(), "verify_category")
            .when(F.col("price").isNull() | (F.col("price") <= 0), "verify_price")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_product_price",
            F.when(F.col("price").isNull() | (F.col("price") <= 0), "invalid_price").otherwise("valid_price")
        )
    )

    # Keep only valid product rows for the transformed Silver table.
    products_good = products_validated.filter(
        (F.col("to_be_verified_by_products_team") == "No Issues") &
        (F.col("check_product_price") == "valid_price")
    )

    if "price_raw" in products_good.columns:
        # Keep only valid product rows for the transformed Silver table.
        products_good = products_good.drop("price_raw")

    # Send invalid product rows to the quarantine dataset for manual review.
    products_bad = products_validated.filter(
        (F.col("to_be_verified_by_products_team") != "No Issues") |
        (F.col("check_product_price") == "invalid_price")
    ).withColumn("quarantine_ts", F.current_timestamp())

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(products_good, "adb_datatocrunchproject.silver_schema.products_transformed", "product_id")
    
    # Append bad product rows to the quarantine table instead of losing them.
    products_bad.write.format("delta").mode("append").saveAsTable("adb_datatocrunchproject.silver_schema.products_quarantine")

    mx_ingested = products_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = products_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested)).agg(F.max("bronze_run_id").alias("mx")).collect()[0]["mx"]

    upsert_silver_control("products", mx_run, mx_ingested, products_good.count())
else:
    print("No new products Bronze rows for Silver.")
    upsert_silver_control(
        "products",
        last_products_run_id,
        last_products_ingested_at,
        products_inc_count
    )


In [0]:
%sql
select * from adb_datatocrunchproject.silver_schema.products_cleaned

## Step 6 — Payments incremental processing
This cell processes payments from Bronze to Silver.
It cleans:
• payment_status
• paid_amount
• processed_at
Then it validates records, quarantines bad rows, and merges valid rows into the Silver transformed payments table.

In [0]:
# Step 6 - Payments incremental processing
# Read only the Bronze payment rows that Silver has not processed yet.
payments_inc, last_payments_ingested_at, last_payment_run_id = get_incremental_bronze("adb_datatocrunchproject.bronze_schema.payments_raw", "payments")
print("Payments last processed Bronze ingested_at = ", last_payments_ingested_at)
#print("Payments incremental Bronze rows =", payments_inc.count())

# Count the incremental payment rows entering Silver in this run.
payments_inc_count = payments_inc.count()
print(f"payments rows_to_process_in_silver = {payments_inc_count}")

if payments_inc.count() > 0:
    # Create a window that keeps the latest payment record for each payment_id.
    payment_window = Window.partitionBy("payment_id").orderBy(
        F.col("processed_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    # Start the Silver payment-cleaning pipeline. This block standardizes and deduplicates raw payment records.
    payments_cleaned = (
        payments_inc
        .withColumn("payment_status", F.upper(F.trim(F.col("payment_status"))))
        .withColumn("payment_status", F.when(F.col("payment_status") == "", F.lit(None)).otherwise(F.col("payment_status")))
        # Start cleaning the payment amount field before converting it to numeric.
        .withColumn("paid_amount", F.trim(F.col("paid_amount")))
        .withColumn("paid_amount", F.regexp_replace(F.col("paid_amount"), r"\$", ""))
        .withColumn("paid_amount", F.regexp_replace(F.col("paid_amount"), r",", ""))
        .withColumn("paid_amount", F.regexp_replace(F.col("paid_amount"), r" ", ""))
        .withColumn("paid_amount", F.expr("try_cast(paid_amount as double)"))
        .withColumn("processed_at", F.to_timestamp("processed_at"))
        # Assign a row number inside each business key so we can keep only the latest version of that record.
        .withColumn("row_rank", F.row_number().over(payment_window))
        # Keep only the latest record for each business key.
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(payments_cleaned, "adb_datatocrunchproject.silver_schema.payments_cleaned", "payment_id")

    # Apply Silver data-quality rules to the cleaned payment records.
    payments_validated = (
        payments_cleaned
        .withColumn(
            "to_be_verified_by_payments_team",
            F.when(F.col("order_id").isNull(), "verify_order_id")
            .when(F.col("payment_status").isNull(), "verify_payment_status")
            .when(F.col("paid_amount").isNull() | (F.col("paid_amount") <= 0), "verify_paid_amount")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_paid_amount",
            F.when(F.col("paid_amount").isNull() | (F.col("paid_amount") <= 0), F.lit(True)).otherwise(F.lit(False))
        )
    )

    # Keep only valid payment rows for the transformed Silver table.
    payments_good = payments_validated.filter(F.col("to_be_verified_by_payments_team") == "No Issues")

    # Send invalid payment rows to the quarantine dataset for manual review.
    payments_bad = payments_validated.filter(F.col("to_be_verified_by_payments_team") != "No Issues").withColumn("quarantine_ts", F.current_timestamp())

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(payments_good, "adb_datatocrunchproject.silver_schema.payments_transformed", "payment_id")

    # Append bad payment rows to the quarantine table instead of losing them.
    payments_bad.write.format("delta").mode("append").saveAsTable("adb_datatocrunchproject.silver_schema.payments_quarantine")

    mx_ingested = payments_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = payments_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested)).agg(F.max("bronze_run_id").alias("mx")).collect()[0]["mx"]

    upsert_silver_control("payments", mx_run, mx_ingested, payments_good.count())
else:
    print("No new payments Bronze rows for Silver.")
    upsert_silver_control(
        "payments",
        last_payment_run_id,
        last_payments_ingested_at,
        payments_inc_count
    )


Step 7 — Quick validation
This final cell prints Silver transformed row counts and shows the Silver control table so you can confirm the incremental processing behavior.


In [0]:
print("Products transformed count :", spark.sql("SELECT COUNT(*) FROM adb_datatocrunchproject.silver_schema.products_transformed").collect()[0][0])
print("Orders transformed count :", spark.sql("SELECT COUNT(*) FROM adb_datatocrunchproject.silver_schema.orders_transformed").collect()[0][0])
print("Payments transformed count :", spark.sql("SELECT COUNT(*) FROM adb_datatocrunchproject.silver_schema.payments_transformed").collect()[0][0])
display(
    spark.table("adb_datatocrunchproject.silver_schema.processing_control")
    .orderBy("entity_name", F.col("updated_at").desc())
)
